In [57]:
using LowLevelFEM, LinearAlgebra

In [58]:
openGeometry("boxes.geo")

In [59]:
#openPreProcessor()

In [60]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [61]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 2000)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

  0.702303 seconds (115.76 k allocations: 72.105 MiB, 1.57% gc time, 18.33% compilation time)


0

In [62]:
contact_pair = contact(u, master="master", slave="slave", cn=1e8)

Contact("slave" -> "master", 1438 candidate nodes, 743 active, G=(4314, 12765), C=(4314, 4314))

In [63]:
support = [bc_bottom, bc_top]
free = freeDoFs(U, support)

u_it = copy(u)

old_tags = copy(contact_pair.master_element_tags)
old_G = copy(contact_pair.G)

for iter in 1:40

    updateContact!(contact_pair, u_it)

    (; G, C, g) = contact_pair

    nchanged = count(old_tags .!= contact_pair.master_element_tags)

    dG = norm(G - old_G) /
         max(norm(old_G), eps())

    println(
        "master changes = ", nchanged,
        ", dG = ", dG
    )

    old_tags = copy(contact_pair.master_element_tags)
    old_G = copy(G)

    # Penalty contact
    p  = -C * g
    rc = -G' * p
    Kc =  G' * C * G

    # Equilibrium residual and tangent
    r = K * u_it - f + rc
    A = K + Kc

    # Homogeneous Newton correction on prescribed DoFs
    Δu = vectorField(U, "body", [0, 0, 0])
    DoFs(Δu)[free] = -A[free, free] \ DoFs(r)[free]

    r0 = norm(DoFs(r)[free])

    α = 1.0
    u_trial = copy(u_it)
    r_trial = nothing

    while α > 1e-6

        u_trial = u_it + α * Δu

        updateContact!(contact_pair, u_trial)

        (; G, C, g) = contact_pair

        p_trial = -C * g
        rc_trial = -G' * p_trial

        r_trial = K * u_trial - f + rc_trial

        if norm(DoFs(r_trial)[free]) < r0
            break
        end

        α *= 0.5
    end

    u_it = copy(u_trial)
    r = r_trial

    err = α * norm(DoFs(Δu)[free]) /
          max(norm(DoFs(u_it)), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(contact_pair.active),
        ", min gap = ", minimum(contact_pair.gap_values),
        ", error = ", err,
        ", |r| = ", norm(DoFs(r)[free])
    )

    err < 1e-8 && break
end

u = u_it

master changes = 0, dG = 1.033040622932867e-16
iter = 1, α = 1.0, active = 213, min gap = -4.173478511617618e-5, error = 0.27124745634464764, |r| = 22933.019141434263
master changes = 6, dG = 0.055885746638950115
iter = 2, α = 0.000244140625, active = 345, min gap = -4.172560647661214e-5, error = 2.0843659031657263e-5, |r| = 20768.93289930721
master changes = 0, dG = 0.00026733320329212323
iter = 3, α = 0.00048828125, active = 441, min gap = -4.170718910875267e-5, error = 1.716795477775611e-5, |r| = 19754.287397677075
master changes = 0, dG = 8.59358204266432e-5
iter = 4, α = 0.0009765625, active = 536, min gap = -4.1670357396558975e-5, error = 1.9331727666862426e-5, |r| = 19137.184287704327
master changes = 0, dG = 3.795829915610555e-5
iter = 5, α = 0.0009765625, active = 571, min gap = -4.1633532536534956e-5, error = 1.3844580685529763e-5, |r| = 19009.859545600873
master changes = 1, dG = 0.024022293352385666
iter = 6, α = 0.0009765625, active = 581, min gap = -4.159681130755117e-5, 

nodal VectorField
[0.0; 0.0; … ; -0.03473347096146808; -0.051258699241181364;;]

In [64]:
showDoFResults(u, name="u cont.", visible=true, factor=1)

1

In [65]:
showElementResults(contact_pair.gap, name="gap")

2

In [66]:
openPostProcessor()

Két vagy több párnál majd:

```Julia
contacts = ContactSet(c1, c2, c3)

updateContact!(contacts, u_it)

Kc = sum(c.G' * c.C * c.G for c in contacts)
rc = sum(c.G' * c.C * c.g for c in contacts)

r = K * u_it - f + rc
A = K + Kc
```